# Semana 4: Modelado Predictivo – SARIMA, Holt-Winters y Prophet

## Proyecto: Serie de Tiempo de Caudal – Estación Pueblo Nuevo (IDEAM)

**Objetivo:** Entrenar y comparar tres modelos de pronóstico de series temporales para predecir el caudal medio diario a 30 días, y responder la pregunta de investigación formulada en la Semana 3.

### Pregunta de Investigación:
> *¿Es posible pronosticar el caudal medio diario de la estación Pueblo Nuevo con un horizonte de 30 días, utilizando modelos de series temporales (SARIMA, Holt-Winters, Prophet) y aprovechando la estacionalidad anual identificada?*

### Contenido:
1. Importar librerías
2. Cargar serie limpia y resumen
3. División Train / Test
4. Modelo 1: SARIMA
5. Modelo 2: Holt-Winters (Exponential Smoothing)
6. Modelo 3: Prophet
7. Comparación visual de pronósticos
8. Métricas de evaluación (RMSE, MAE, MAPE)
9. Pronóstico a futuro (30 días más allá de los datos)
10. Respuesta a la pregunta de investigación
11. Conclusiones del proyecto

## 1. Importar Librerías

In [70]:

# =============================================================================
# CONCEPTO: Importación de librerías para modelado predictivo de series de tiempo
# -----------------------------------------------------------------------------
# pandas / numpy        → manipulación de datos y cálculos numéricos
# plotly.express / graph_objects / make_subplots → visualizaciones interactivas
#
# statsmodels.tsa.statespace.sarimax.SARIMAX
#   → modelo SARIMA(p,d,q)(P,D,Q)s con soporte para variables exógenas (X)
#   → implementación en espacio de estados; más estable numéricamente que ARIMA clásico
#
# statsmodels.tsa.holtwinters.ExponentialSmoothing
#   → modelo de suavización exponencial triple (Holt-Winters)
#   → soporta tendencia y estacionalidad aditiva o multiplicativa
#
# prophet.Prophet
#   → modelo aditivo bayesiano de Meta (Facebook) para series temporales
#   → requiere formato específico: columnas "ds" (fecha) y "y" (valor)
#   → detecta automáticamente estacionalidades y puntos de cambio en tendencia
#
# sklearn.metrics
#   - mean_squared_error        → MSE; raíz = RMSE (penaliza errores grandes)
#   - mean_absolute_error       → MAE (interpretable en unidades originales)
#   - mean_absolute_percentage_error → MAPE (porcentual, independiente de escala)
#
# warnings.filterwarnings("ignore")
#   → suprime avisos de convergencia de optimizadores numéricos (SARIMA, HW)
#
# CRITERIO DE USO: Este conjunto de librerías cubre los tres paradigmas de
# pronóstico de series temporales: modelos ARIMA (statsmodels), suavización
# exponencial (statsmodels) y modelos de tendencia-estacionalidad descompuesta
# (Prophet). Se importan juntos para facilitar la comparación directa.
# =============================================================================
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Modelos de series de tiempo
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from prophet import Prophet

# Métricas
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

import warnings
warnings.filterwarnings("ignore")

import plotly.io as pio
pio.templates.default = "plotly_white"

print("✅ Librerías importadas")


✅ Librerías importadas


## 2. Cargar Serie Limpia y Resumen

Usamos el CSV `caudal_limpio_diario.csv` exportado en la Semana 2, que contiene la serie diaria completa (imputada + capped).

In [61]:

# =============================================================================
# CONCEPTO: Carga y preparación de la serie para modelado predictivo
# -----------------------------------------------------------------------------
# pd.read_csv(ruta, index_col="Fecha", parse_dates=True)
#   → carga la serie exportada en la Semana 2 con DatetimeIndex
#   → ruta relativa "../Week_2/..." → portabilidad entre entornos
#
# df.asfreq("D")
#   → fuerza la frecuencia explícita a "D" (diaria)
#   → CRÍTICO para SARIMA y Holt-Winters: sin frecuencia definida,
#     statsmodels lanza warnings o errores al ajustar el modelo
#   → si hay fechas faltantes, asfreq("D") inserta NaN (no debe ocurrir
#     en nuestra serie ya imputada en Semana 2)
#
# Verificaciones previas al modelado:
#   - NaN = 0 → ningún modelo tolera valores faltantes sin manejo especial
#   - Freq = "D" → confirma que el índice tiene frecuencia regular
#   - Período completo → define el horizonte disponible para train/test
#
# display(df["Caudal"].describe().to_frame().T)
#   → transpone el describe() para mostrar estadísticos en una sola fila
#   → facilita la lectura de min/max/mean antes de dividir los datos
#
# CRITERIO DE USO: Verificar siempre la frecuencia del índice antes de
# entrenar modelos de series de tiempo. Un índice sin frecuencia o con
# fechas irregulares puede causar errores silenciosos en las predicciones.
# =============================================================================
# Cargar serie limpia
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DATOS/SEMANA4/caudal_eda_avanzado.csv", index_col="Fecha", parse_dates=True)

# Asegurar frecuencia diaria
df = df.asfreq("D")

print(f"✅ Serie cargada")
print(f"   Período:  {df.index.min().date()} → {df.index.max().date()}")
print(f"   Registros: {len(df)}")
print(f"   NaN:       {df['Caudal'].isna().sum()}")
print(f"   Freq:      {df.index.freq}")
print(f"\n📊 Estadísticas:")
display(df["Caudal"].describe().to_frame().T.round(2))


✅ Serie cargada
   Período:  2010-01-01 → 2017-12-31
   Registros: 2922
   NaN:       0
   Freq:      <Day>

📊 Estadísticas:


,count,mean,std,min,25%,50%,75%,max
Caudal,2922.0,3.38,1.46,0.15,2.74,3.33,3.95,8.8


In [62]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. División Train / Test

Reservamos los **últimos 60 días** como conjunto de prueba (test).  
- **Train:** Todo excepto los últimos 60 días → para entrenar los modelos  
- **Test:** Últimos 60 días → para evaluar precisión  
- **Horizonte de pronóstico:** 60 días (evaluamos en test; luego pronosticamos 30 días a futuro)

In [63]:

# =============================================================================
# CONCEPTO: División temporal Train / Test (Hold-Out)
# -----------------------------------------------------------------------------
# HORIZONTE = 60  → reservamos los últimos 60 días como conjunto de prueba
#   Criterio de elección: 60 días > horizonte de pronóstico práctico (30 d)
#   y representa ~2 meses = suficiente para evaluar el patrón estacional
#   a corto plazo sin sacrificar demasiados datos de entrenamiento.
#
# train = df["Caudal"].iloc[:-HORIZONTE]
#   → todos los datos excepto los últimos 60 → entrenamiento del modelo
#   → iloc con índice negativo: -N toma desde el inicio hasta N antes del final
#
# test = df["Caudal"].iloc[-HORIZONTE:]
#   → los últimos 60 días → evaluación out-of-sample (no vistos en entrenamiento)
#
# División Hold-Out vs Validación Cruzada Temporal:
#   Hold-Out:                  simple, rápido, 1 evaluación
#   Walk-Forward Validation:   múltiples ventanas, más robusta pero costosa
#   Cross-validation temporal: requiere TimeSeriesSplit de sklearn
#   → Para este proyecto usamos Hold-Out por simplicidad computacional
#
# add_vrect(x0, x1, fillcolor) → resalta visualmente el período de test
#   con un fondo semitransparente rojo, facilitando la identificación del
#   período de evaluación en el gráfico
#
# CRITERIO DE USO: En series de tiempo NO se mezclan aleatoriamente train
# y test como en modelos tabulares. El test SIEMPRE debe ser el período
# más reciente para respetar la causalidad temporal y evitar data leakage.
# =============================================================================
# División train/test
HORIZONTE = 60  # días de test

train = df["Caudal"].iloc[:-HORIZONTE]
test = df["Caudal"].iloc[-HORIZONTE:]

print(f"📊 División Train / Test:")
print(f"   Train: {train.index.min().date()} → {train.index.max().date()} ({len(train)} días)")
print(f"   Test:  {test.index.min().date()} → {test.index.max().date()} ({len(test)} días)")

# Visualización del split
fig_split = go.Figure()
fig_split.add_trace(go.Scatter(x=train.index, y=train, mode="lines",
              name="Train", line=dict(color="#1f77b4", width=0.8)))
fig_split.add_trace(go.Scatter(x=test.index, y=test, mode="lines",
              name="Test", line=dict(color="#d62728", width=1.5)))
fig_split.add_vrect(x0=test.index.min(), x1=test.index.max(),
              fillcolor="rgba(255,0,0,0.05)", line_width=0,
              annotation_text="Test (60 días)", annotation_position="top left")
fig_split.update_layout(title="División Train / Test", xaxis_title="",
                  yaxis_title="Caudal (m³/s)", width=1000, height=450,
                  hovermode="x unified")
fig_split.show()


📊 División Train / Test:
   Train: 2010-01-01 → 2017-11-01 (2862 días)
   Test:  2017-11-02 → 2017-12-31 (60 días)


## 4. Modelo 1: SARIMA

**SARIMA** (*Seasonal AutoRegressive Integrated Moving Average*) extiende ARIMA con componentes estacionales:

$$SARIMA(p, d, q)(P, D, Q)_s$$

- $(p, d, q)$: Parte no estacional (AR, diferenciación, MA)
- $(P, D, Q)_s$: Parte estacional con período $s$

Dado que el período estacional anual (365) es demasiado grande para SARIMA, usamos **resampling mensual** o un período estacional de **30 días** (aproximación mensual) para mantener la viabilidad computacional.

> **Nota:** Usaremos `order=(1,1,1)` y `seasonal_order=(1,1,1,30)` como punto de partida basado en el ACF/PACF de la Semana 3.

In [67]:

%%time
# =============================================================================
# CONCEPTO: Entrenamiento del modelo SARIMA(1,1,1)(1,1,1)₃₀
# -----------------------------------------------------------------------------
# SARIMAX(endog, order, seasonal_order, enforce_stationarity, enforce_invertibility)
#   - endog               → serie endógena de entrenamiento (train)
#   - order=(p, d, q)     → parte no estacional:
#       p=1: un término autorregresivo AR(1) → el valor actual depende de Yt-1
#       d=1: una diferenciación → elimina raíz unitaria (serie I(1) según ADF/KPSS)
#       q=1: un término de media móvil MA(1) → corrige el error del paso anterior
#   - seasonal_order=(P, D, Q, s):
#       P=1, D=1, Q=1: términos estacionales AR, diferenciación y MA
#       s=30: período estacional en días → aproximación mensual
#       NOTA: s=365 (anual real) es computacionalmente inviable para datos diarios
#              por la dimensión de la matriz de covarianza (365×365)
#   - enforce_stationarity=False → no transforma los parámetros AR automáticamente
#   - enforce_invertibility=False → no transforma los parámetros MA automáticamente
#     → Ambos en False: da mayor flexibilidad al optimizador numérico
#
# .fit(disp=False, maxiter=200)
#   - disp=False  → suprime la impresión de iteraciones del optimizador
#   - maxiter=200 → máximo de iteraciones del solver L-BFGS-B
#
# resultado_sarima.aic / .bic
#   → criterios de información para comparar modelos de igual familia:
#   AIC = 2k - 2ln(L)    → penaliza número de parámetros k
#   BIC = k·ln(N) - 2ln(L) → penaliza más fuerte que AIC en N grande
#   Menor AIC/BIC → mejor ajuste con menos complejidad
#
# %%time → magic command de Jupyter: mide el tiempo total de ejecución de la celda
#
# CRITERIO DE USO: SARIMA es la extensión natural de ARIMA para series con
# estacionalidad. Los órdenes (p,d,q)(P,D,Q,s) se determinan mediante el
# ACF/PACF (Semana 3) o se seleccionan automáticamente con auto_arima (pmdarima).
# =============================================================================
# Entrenar SARIMA
# order=(p,d,q), seasonal_order=(P,D,Q,s)
# s=30 como aproximación mensual (365 es computacionalmente inviable)
print("🔄 Entrenando SARIMA(1,1,1)(1,1,1,30)...")

modelo_sarima = SARIMAX(
    train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 30),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
resultado_sarima = modelo_sarima.fit(disp=False, maxiter=200)

print(f"\n✅ SARIMA entrenado")
print(f"   AIC: {resultado_sarima.aic:.2f}")
print(f"   BIC: {resultado_sarima.bic:.2f}")
print(f"\n📋 Resumen de parámetros:")
print(resultado_sarima.summary().tables[1])


🔄 Entrenando SARIMA(1,1,1)(1,1,1,30)...

✅ SARIMA entrenado
   AIC: 5325.59
   BIC: 5355.28

📋 Resumen de parámetros:
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.5812      0.014     42.917      0.000       0.555       0.608
ma.L1         -0.8704      0.010    -84.214      0.000      -0.891      -0.850
ar.S.L30      -0.0361      0.021     -1.699      0.089      -0.078       0.006
ma.S.L30      -1.0202      0.009   -107.911      0.000      -1.039      -1.002
sigma2         0.3632      0.006     62.341      0.000       0.352       0.375
CPU times: user 47.7 s, sys: 498 ms, total: 48.2 s
Wall time: 43.2 s


In [73]:
%%time
# =============================================================================
# CONCEPTO: Entrenamiento y Pronóstico SARIMA sobre el período test y diagnóstico de residuos
# -----------------------------------------------------------------------------
# Consolidamos el entrenamiento del modelo SARIMA y su pronóstico/evaluación
# en una sola celda para asegurar que `resultado_sarima` y las métricas
# `rmse_s`, `mae_s`, `mape_s` estén siempre disponibles.
# =============================================================================
# Entrenar SARIMA (código movido de la celda d3504ae3)
# order=(p,d,q), seasonal_order=(P,D,Q,s)
# s=30 como aproximación mensual (365 es computacionalmente inviable)
print("🔄 Entrenando SARIMA(1,1,1)(1,1,1,30)...")

modelo_sarima = SARIMAX(
    train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 30),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
resultado_sarima = modelo_sarima.fit(disp=False, maxiter=200)

print(f"\n✅ SARIMA entrenado")
print(f"   AIC: {resultado_sarima.aic:.2f}")
print(f"   BIC: {resultado_sarima.bic:.2f}")
print(f"\n📋 Resumen de parámetros:")
print(resultado_sarima.summary().tables[1])


# Pronóstico SARIMA sobre el período de test
pred_sarima = resultado_sarima.forecast(steps=HORIZONTE)
pred_sarima.index = test.index

# Diagnóstico de residuos
residuos_sarima = resultado_sarima.resid

fig_sarima_diag = make_subplots(rows=1, cols=2,
                    subplot_titles=["Pronóstico SARIMA vs Real", "Residuos del modelo"])

fig_sarima_diag.add_trace(go.Scatter(x=test.index, y=test, mode="lines",
              name="Real", line=dict(color="#1f77b4", width=2)), row=1, col=1)
fig_sarima_diag.add_trace(go.Scatter(x=test.index, y=pred_sarima, mode="lines",
              name="SARIMA", line=dict(color="#ff7f0e", width=2, dash="dash")), row=1, col=1)

fig_sarima_diag.add_trace(go.Histogram(x=residuos_sarima, nbinsx=50,
              marker_color="#ff7f0e", name="Residuos"), row=1, col=2)

fig_sarima_diag.update_layout(title="Modelo SARIMA(1,1,1)(1,1,1,30)",
                  width=1000, height=400, showlegend=True)
fig_sarima_diag.update_yaxes(title_text="Caudal (m³/s)", row=1, col=1)
fig_sarima_diag.show()

rmse_s = np.sqrt(mean_squared_error(test, pred_sarima))
mae_s = mean_absolute_error(test, pred_sarima)
mape_s = mean_absolute_percentage_error(test, pred_sarima) * 100
print(f"\n📊 SARIMA – Métricas en Test:")
print(f"   RMSE: {rmse_s:.2f} m³/s")
print(f"   MAE:  {mae_s:.2f} m³/s")
print(f"   MAPE: {mape_s:.1f}%")

🔄 Entrenando SARIMA(1,1,1)(1,1,1,30)...

✅ SARIMA entrenado
   AIC: 5325.59
   BIC: 5355.28

📋 Resumen de parámetros:
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.5812      0.014     42.917      0.000       0.555       0.608
ma.L1         -0.8704      0.010    -84.214      0.000      -0.891      -0.850
ar.S.L30      -0.0361      0.021     -1.699      0.089      -0.078       0.006
ma.S.L30      -1.0202      0.009   -107.911      0.000      -1.039      -1.002
sigma2         0.3632      0.006     62.341      0.000       0.352       0.375



📊 SARIMA – Métricas en Test:
   RMSE: 0.49 m³/s
   MAE:  0.44 m³/s
   MAPE: 17.9%
CPU times: user 47.5 s, sys: 523 ms, total: 48.1 s
Wall time: 43.1 s


## 5. Modelo 2: Holt-Winters (Triple Exponential Smoothing)

**Holt-Winters** descompone la serie en nivel, tendencia y estacionalidad, actualizando cada componente con suavizamiento exponencial.

$$\hat{Y}_{t+h} = (l_t + h \cdot b_t) \cdot s_{t+h-m}$$

- Usamos la variante **aditiva** para estacionalidad y tendencia
- Período estacional: **30 días** (ciclo mensual como aproximación)

In [74]:
Este es el modelo Holt-Winters (Suavización Exponencial Triple), que descompone la serie en nivel, tendencia y estacionalidad.

SyntaxError: invalid syntax (4160198801.py, line 1)

In [75]:
%%time
# =============================================================================
# CONCEPTO: Entrenamiento y Pronóstico Holt-Winters en el período test y cálculo de métricas
# -----------------------------------------------------------------------------
# Consolidamos el entrenamiento del modelo Holt-Winters y su pronóstico/evaluación
# en una sola celda para asegurar que `resultado_hw` y las métricas
# `rmse_hw`, `mae_hw`, `mape_hw` estén siempre disponibles.
# =============================================================================
# Entrenar Holt-Winters (código movido de la celda 4f98091e)
print("🔄 Entrenando Holt-Winters (aditivo, período=30)...")

modelo_hw = ExponentialSmoothing(
    train,
    trend="add",
    seasonal="add",
    seasonal_periods=30,
    initialization_method="estimated",
)
resultado_hw = modelo_hw.fit(optimized=True)

print(f"\n✅ Holt-Winters entrenado")
print(f"   AIC: {resultado_hw.aic:.2f}")
print(f"   Parámetros optimizados:")
print(f"     α (nivel):         {resultado_hw.params['smoothing_level']:.4f}")
print(f"     β (tendencia):     {resultado_hw.params['smoothing_trend']:.4f}")
print(f"     γ (estacionalidad): {resultado_hw.params['smoothing_seasonal']:.4f}")


# Pronóstico Holt-Winters
pred_hw = resultado_hw.forecast(HORIZONTE)
pred_hw.index = test.index

fig_hw_test = go.Figure()
fig_hw_test.add_trace(go.Scatter(x=test.index, y=test, mode="lines",
              name="Real", line=dict(color="#1f77b4", width=2)))
fig_hw_test.add_trace(go.Scatter(x=test.index, y=pred_hw, mode="lines",
              name="Holt-Winters", line=dict(color="#2ca02c", width=2, dash="dash")))
fig_hw_test.update_layout(title="Pronóstico Holt-Winters vs Real",
                  xaxis_title="", yaxis_title="Caudal (m³/s)",
                  width=900, height=400, hovermode="x unified")
fig_hw_test.show()

rmse_hw = np.sqrt(mean_squared_error(test, pred_hw))
mae_hw = mean_absolute_error(test, pred_hw)
mape_hw = mean_absolute_percentage_error(test, pred_hw) * 100
print(f"\n📊 Holt-Winters – Métricas en Test:")
print(f"   RMSE: {rmse_hw:.2f} m³/s")
print(f"   MAE:  {mae_hw:.2f} m³/s")
print(f"   MAPE: {mape_hw:.1f}%")

🔄 Entrenando Holt-Winters (aditivo, período=30)...

✅ Holt-Winters entrenado
   AIC: -2600.68
   Parámetros optimizados:
     α (nivel):         0.7002
     β (tendencia):     0.0000
     γ (estacionalidad): 0.0000



📊 Holt-Winters – Métricas en Test:
   RMSE: 0.57 m³/s
   MAE:  0.52 m³/s
   MAPE: 21.3%
CPU times: user 1.71 s, sys: 3.94 ms, total: 1.71 s
Wall time: 1.68 s


## 6. Modelo 3: Prophet

**Prophet** (Meta/Facebook) es un modelo aditivo diseñado para series con:
- Estacionalidad fuerte (anual, semanal)
- Tendencia no lineal
- Datos faltantes y outliers

$$y(t) = g(t) + s(t) + h(t) + \epsilon_t$$

Donde $g(t)$ es tendencia, $s(t)$ estacionalidad, $h(t)$ efectos de feriados.

Prophet detecta automáticamente la estacionalidad anual — ideal para nuestros datos hidrológicos.

In [ ]:
Este es el modelo Prophet (Meta/Facebook), diseñado para series con estacionalidad fuerte, tendencia no lineal, y datos faltantes/outliers.

In [76]:
%%time
# =============================================================================
# CONCEPTO: Entrenamiento del modelo Prophet y generación del pronóstico + Métricas
# -----------------------------------------------------------------------------
# Consolidamos el entrenamiento y pronóstico de Prophet, junto con el cálculo
# de sus métricas, en una sola celda para asegurar que `pred_prophet` y las métricas
# `rmse_pr`, `mae_pr`, `mape_pr` estén siempre disponibles.
# =============================================================================
# Preparar datos para Prophet (requiere columnas 'ds' y 'y')
df_prophet_train = train.reset_index().rename(columns={"Fecha": "ds", "Caudal": "y"})

# Entrenar Prophet (código movido de la celda d0355a55)
print("🔄 Entrenando Prophet...")
modelo_prophet = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    changepoint_prior_scale=0.05,
    seasonality_mode="additive",
)
modelo_prophet.fit(df_prophet_train)

# Crear DataFrame futuro para pronóstico
futuro = modelo_prophet.make_future_dataframe(periods=HORIZONTE, freq="D")
pronostico_prophet = modelo_prophet.predict(futuro)

# Extraer solo el período de test
pred_prophet = pronostico_prophet.set_index("ds").loc[test.index, "yhat"]

print(f"\n✅ Prophet entrenado")
print(f"   Puntos de cambio detectados: {len(modelo_prophet.changepoints)}")


# Métricas Prophet
rmse_pr = np.sqrt(mean_squared_error(test, pred_prophet))
mae_pr  = mean_absolute_error(test, pred_prophet)
mape_pr = np.mean(np.abs((test - pred_prophet) / test)) * 100

print("=" * 50)
print("📊 Métricas Prophet")
print("=" * 50)
print(f"   RMSE : {rmse_pr:.4f} m³/s")
print(f"   MAE  : {mae_pr:.4f} m³/s")
print(f"   MAPE : {mape_pr:.2f} %")

# Visualización Prophet
fig_prophet_test = go.Figure()
fig_prophet_test.add_trace(go.Scatter(
    x=train.index[-120:], y=train.iloc[-120:],
    mode="lines", name="Train (últimos 120 d)",
    line=dict(color="#636EFA")
))
fig_prophet_test.add_trace(go.Scatter(
    x=test.index, y=test,
    mode="lines", name="Real (Test)",
    line=dict(color="#EF553B", width=2)
))
fig_prophet_test.add_trace(go.Scatter(
    x=test.index, y=pred_prophet,
    mode="lines", name="Prophet",
    line=dict(color="#00CC96", dash="dash", width=2)
))
fig_prophet_test.update_layout(
    title="Prophet – Pronóstico vs Real",
    xaxis_title="Fecha", yaxis_title="Caudal (m³/s)",
    template="plotly_white", height=450,
    legend=dict(orientation="h", yanchor="bottom", y=1.02)
)
fig_prophet_test.show()

🔄 Entrenando Prophet...

✅ Prophet entrenado
   Puntos de cambio detectados: 25
📊 Métricas Prophet
   RMSE : 1.4734 m³/s
   MAE  : 1.4146 m³/s
   MAPE : 57.44 %


CPU times: user 1.94 s, sys: 85.6 ms, total: 2.02 s
Wall time: 2.87 s


## 7. Comparación Visual de los Tres Modelos

Superponemos los pronósticos de **SARIMA**, **Holt-Winters** y **Prophet** sobre los datos reales del período de prueba para evaluar visualmente cuál modelo captura mejor la dinámica del caudal.

In [77]:

# =============================================================================
# CONCEPTO: Comparación visual superpuesta de los tres modelos de pronóstico
# -----------------------------------------------------------------------------
# Objetivo: un único gráfico que permite evaluar visualmente cuál modelo
# sigue mejor la dinámica real del caudal en el período de prueba.
#
# Codificación visual por modelo:
#   dash="dash"    → SARIMA   (rojo #EF553B)
#   dash="dot"     → Holt-Winters (verde #00CC96)
#   dash="dashdot" → Prophet  (morado #AB63FA)
#   línea sólida negra (width=2.5) → datos reales (referencia más gruesa)
#
# Contexto de train (últimos 120 días, opacity=0.5):
#   → muestra la continuidad entre datos históricos y pronóstico
#   → opacity reducida para que el foco visual quede en el test
#
# Leyenda con RMSE en el nombre de cada traza:
#   name=f"SARIMA  (RMSE={rmse_s:.2f})"
#   → integra la métrica directamente en la leyenda para comparación rápida
#   → evita la necesidad de consultar una tabla separada
#
# add_shape(type="line") → línea vertical en el inicio del test
#   x0=x1=test.index[0]: posición temporal del corte train/test
#   y0=0, y1=1, yref="paper" → línea que ocupa el 100% de la altura del gráfico
#   independientemente de la escala del eje Y
#
# add_annotation → etiqueta textual asociada a la línea vertical
#   showarrow=False: sin flecha, solo texto flotante
#   yanchor="bottom", y=1: posicionado en la parte superior del gráfico
#
# legend(orientation="h", y=1.05) → leyenda horizontal encima del gráfico
#   → evita que tape las series cuando hay múltiples trazas superpuestas
#
# CRITERIO DE USO: La comparación visual es el primer filtro antes de elegir
# un modelo. Un modelo numéricamente mejor (menor RMSE) pero que visualmente
# diverge en la tendencia puede indicar un problema de especificación.
# La inspección visual y las métricas deben usarse conjuntamente.
# =============================================================================
fig_compare_all = go.Figure()

# Datos reales – train (últimos 120 d) + test
fig_compare_all.add_trace(go.Scatter(
    x=train.index[-120:], y=train.iloc[-120:],
    mode="lines", name="Train (últimos 120 d)",
    line=dict(color="#636EFA", width=1),
    opacity=0.5,
))
fig_compare_all.add_trace(go.Scatter(
    x=test.index, y=test,
    mode="lines", name="Real (Test)",
    line=dict(color="black", width=2.5),
))

# Pronósticos
fig_compare_all.add_trace(go.Scatter(
    x=test.index, y=pred_sarima,
    mode="lines", name=f"SARIMA  (RMSE={rmse_s:.2f})",
    line=dict(color="#EF553B", dash="dash", width=2),
))
fig_compare_all.add_trace(go.Scatter(
    x=test.index, y=pred_hw,
    mode="lines", name=f"Holt-Winters (RMSE={rmse_hw:.2f})",
    line=dict(color="#00CC96", dash="dot", width=2),
))
fig_compare_all.add_trace(go.Scatter(
    x=test.index, y=pred_prophet,
    mode="lines", name=f"Prophet (RMSE={rmse_pr:.2f})",
    line=dict(color="#AB63FA", dash="dashdot", width=2),
))

# Línea vertical separando train/test
fig_compare_all.add_shape(
    type="line",
    x0=test.index[0], x1=test.index[0],
    y0=0, y1=1, yref="paper",
    line=dict(color="gray", dash="longdash", width=1),
)
fig_compare_all.add_annotation(
    x=test.index[0], y=1, yref="paper",
    text="Inicio Test", showarrow=False,
    yanchor="bottom", font=dict(color="gray"),
)

fig_compare_all.update_layout(
    title=dict(text="Comparación de Pronósticos – SARIMA vs Holt-Winters vs Prophet", y=0.98),
    xaxis_title="Fecha", yaxis_title="Caudal (m³/s)",
    template="plotly_white", height=550,
    margin=dict(t=120),
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5),
)
fig_compare_all.show()


## 8. Tabla Comparativa de Métricas

Consolidamos **RMSE**, **MAE** y **MAPE** de los tres modelos para identificar el mejor candidato de pronóstico.

| Métrica | Definición |
|---------|-----------|
| **RMSE** | Raíz del error cuadrático medio — penaliza errores grandes |
| **MAE**  | Error absoluto medio — interpretación directa en m³/s |
| **MAPE** | Error porcentual absoluto medio — independiente de escala |

In [78]:

# =============================================================================
# CONCEPTO: Tabla comparativa de métricas y gráfico de barras agrupadas
# -----------------------------------------------------------------------------
# pd.DataFrame con los 3 modelos y sus 3 métricas → estructura tabular comparativa
#   .set_index("Modelo") → cada fila es un modelo; columnas son las métricas
#
# metricas.idxmin()
#   → devuelve el nombre del índice (modelo) con el valor mínimo por cada columna
#   → identifica el mejor modelo en RMSE, MAE y MAPE independientemente
#   → si el mismo modelo lidera en las 3 métricas: consenso claro
#   → si modelos distintos lideran en métricas distintas: trade-off a considerar
#
# .style.highlight_min(axis=0, color="#d4edda")
#   → resalta en verde claro (#d4edda) el valor mínimo de cada columna
#   → axis=0: aplica la comparación por columna (entre modelos)
#   → renderizado HTML en Jupyter: tabla visualmente clara para comunicación
#
# go.Bar con barmode="group"
#   → barras agrupadas (side-by-side): un grupo por métrica, una barra por modelo
#   → permite comparar los tres modelos en las 3 métricas simultáneamente
#   → text=metricas.loc[modelo].round(2) + textposition="outside"
#     → etiquetas numéricas encima de cada barra para lectura precisa
#
# Distinción conceptual de las métricas:
#   RMSE → en m³/s; penaliza errores grandes (cuadrático) → sensible a outliers
#   MAE  → en m³/s; menos sensible a valores extremos, más robusto
#   MAPE → en %; escala independiente, permite comparar entre estaciones o series
#
# CRITERIO DE USO: Reportar siempre las tres métricas juntas: RMSE para
# comunicar error "en unidades", MAPE para comparar proporcionalidad, y MAE
# como medida robusta. El modelo ganador en RMSE no siempre es el mejor en MAPE.
# =============================================================================
# Tabla resumen de métricas
metricas = pd.DataFrame({
    "Modelo": ["SARIMA", "Holt-Winters", "Prophet"],
    "RMSE (m³/s)": [rmse_s, rmse_hw, rmse_pr],
    "MAE (m³/s)":  [mae_s,  mae_hw,  mae_pr],
    "MAPE (%)":    [mape_s, mape_hw, mape_pr],
}).set_index("Modelo")

# Resaltar el mejor modelo por cada métrica
mejor_modelo = metricas.idxmin()
print("🏆 Mejor modelo por métrica:")
for metrica, modelo in mejor_modelo.items():
    print(f"   {metrica}: {modelo}")
print()
display(metricas.round(4).style.highlight_min(axis=0, color="#d4edda"))

# Gráfico de barras agrupadas
fig_metrics_bar = go.Figure()
colores = {"SARIMA": "#EF553B", "Holt-Winters": "#00CC96", "Prophet": "#AB63FA"}

for modelo in metricas.index:
    fig_metrics_bar.add_trace(go.Bar(
        name=modelo,
        x=metricas.columns,
        y=metricas.loc[modelo],
        marker_color=colores[modelo],
        text=metricas.loc[modelo].round(2),
        textposition="outside",
    ))

fig_metrics_bar.update_layout(
    title="Comparación de Métricas de Error",
    barmode="group",
    yaxis_title="Valor",
    template="plotly_white",
    height=450,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5),
)
fig_metrics_bar.show()


🏆 Mejor modelo por métrica:
   RMSE (m³/s): SARIMA
   MAE (m³/s): SARIMA
   MAPE (%): SARIMA



,RMSE (m³/s),MAE (m³/s),MAPE (%)
Modelo,,,
SARIMA,0.492400,0.440300,17.946000
Holt-Winters,0.570000,0.520600,21.297000
Prophet,1.473400,1.414600,57.437500


## 9. Pronóstico a Futuro con el Mejor Modelo

Seleccionamos el modelo con **menor RMSE** y generamos un pronóstico de **30 días** más allá del último dato disponible en la serie.  
Este pronóstico tiene valor práctico para la gestión hídrica de la estación **Pueblo Nuevo**.

In [79]:

# =============================================================================
# CONCEPTO: Pronóstico a futuro con el mejor modelo — reentrenamiento con serie completa
# -----------------------------------------------------------------------------
# HORIZONTE_FUTURO = 30 → 30 días más allá del último dato disponible
#   Horizonte práctico para gestión hídrica (alerta temprana, planificación de riego)
#
# metricas["RMSE (m³/s)"].idxmin()
#   → identifica dinámicamente el mejor modelo según RMSE en test
#   → el código de selección es genérico: funciona independientemente del ganador
#
# Reentrenamiento con la serie COMPLETA (df_full = train + test)
#   Objetivo: aprovechar todos los datos disponibles para el pronóstico final
#   → el modelo entrenado solo en train fue para evaluación;
#     el modelo final se entrena con la serie completa para maximizar información
#
# Lógica condicional por modelo (if/elif/else):
#   SARIMA:       SARIMAX(serie_full, ...).fit() → .forecast(30)
#   Holt-Winters: ExponentialSmoothing(serie_full, ...).fit() → .forecast(30)
#   Prophet:      requiere reset_index + rename; make_future_dataframe(30) →
#                 predict() → .iloc[-HORIZONTE_FUTURO:] extrae los últimos 30 filas
#                 (las fechas futuras, no las históricas)
#
# pd.date_range(start=serie_full.index[-1] + pd.Timedelta(days=1), periods=30, freq="D")
#   → genera las próximas 30 fechas diarias desde el día siguiente al último dato
#   → pd.Timedelta(days=1): desplazamiento de 1 día hacia adelante
#
# futuro_pred.index = fechas_futuro
#   → asigna las fechas reales al pronóstico (cada modelo devuelve índices distintos)
#
# Visualización: serie histórica (últimos 180 d) + pronóstico superpuesto
#   → 180 días de contexto = 6 meses: suficiente para ver la estacionalidad reciente
#   → add_shape + add_annotation: línea vertical en el último dato real
#
# CRITERIO DE USO: El reentrenamiento con todos los datos antes de pronosticar
# es la práctica estándar en producción. La evaluación en test solo sirve para
# seleccionar el modelo; el pronóstico final usa la mayor cantidad de información.
# =============================================================================
HORIZONTE_FUTURO = 30

# Identificar mejor modelo
mejor = metricas["RMSE (m³/s)"].idxmin()
print(f"🏆 Mejor modelo seleccionado: {mejor}\n")

# Reentrenar con TODA la serie y pronosticar 30 días
df_full = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DATOS/SEMANA4/caudal_eda_avanzado.csv", parse_dates=["Fecha"], index_col="Fecha")
df_full = df_full.asfreq("D")
serie_full = df_full["Caudal"]

if mejor == "SARIMA":
    modelo_final = SARIMAX(serie_full, order=(1,1,1), seasonal_order=(1,1,1,30),
                           enforce_stationarity=False, enforce_invertibility=False)
    ajuste_final = modelo_final.fit(disp=False)
    futuro_pred = ajuste_final.forecast(steps=HORIZONTE_FUTURO)

elif mejor == "Holt-Winters":
    modelo_final = ExponentialSmoothing(
        serie_full, trend="add", seasonal="add", seasonal_periods=30
    )
    ajuste_final = modelo_final.fit(optimized=True)
    futuro_pred = ajuste_final.forecast(steps=HORIZONTE_FUTURO)

else:  # Prophet
    df_pr_full = serie_full.reset_index().rename(columns={"Fecha": "ds", "Caudal": "y"})
    modelo_final = Prophet(yearly_seasonality=True, weekly_seasonality=False,
                           daily_seasonality=False, changepoint_prior_scale=0.05,
                           seasonality_mode="additive")
    modelo_final.fit(df_pr_full)
    futuro_df = modelo_final.make_future_dataframe(periods=HORIZONTE_FUTURO, freq="D")
    futuro_all = modelo_final.predict(futuro_df)
    futuro_pred = futuro_all.set_index("ds")["yhat"].iloc[-HORIZONTE_FUTURO:]

# Fechas futuras
fechas_futuro = pd.date_range(start=serie_full.index[-1] + pd.Timedelta(days=1),
                              periods=HORIZONTE_FUTURO, freq="D")
futuro_pred.index = fechas_futuro

print(f"📅 Pronóstico del {fechas_futuro[0].date()} al {fechas_futuro[-1].date()}")
print(f"   Caudal medio pronosticado: {futuro_pred.mean():.2f} m³/s")
print(f"   Rango: [{futuro_pred.min():.2f}, {futuro_pred.max():.2f}] m³/s")

# Visualización
fig_future_forecast = go.Figure()
fig_future_forecast.add_trace(go.Scatter(
    x=serie_full.index[-180:], y=serie_full.iloc[-180:],
    mode="lines", name="Histórico (últimos 180 d)",
    line=dict(color="#636EFA"),
))
fig_future_forecast.add_trace(go.Scatter(
    x=fechas_futuro, y=futuro_pred,
    mode="lines+markers", name=f"Pronóstico {mejor} (30 d)",
    line=dict(color="#EF553B", width=2.5),
    marker=dict(size=4),
))
fig_future_forecast.add_shape(
    type="line",
    x0=serie_full.index[-1], x1=serie_full.index[-1],
    y0=0, y1=1, yref="paper",
    line=dict(color="gray", dash="longdash", width=1),
)
fig_future_forecast.add_annotation(
    x=serie_full.index[-1], y=1, yref="paper",
    text="Último dato real", showarrow=False,
    yanchor="bottom", font=dict(color="gray"),
)

fig_future_forecast.update_layout(
    title=dict(text=f"Pronóstico a 30 Días – Modelo {mejor}", y=0.98),
    xaxis_title="Fecha", yaxis_title="Caudal (m³/s)",
    template="plotly_white", height=500,
    margin=dict(t=80),
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5),
)
fig_future_forecast.show()


🏆 Mejor modelo seleccionado: SARIMA

📅 Pronóstico del 2018-01-01 al 2018-01-30
   Caudal medio pronosticado: 2.44 m³/s
   Rango: [2.29, 2.56] m³/s


## 10. Respuesta a la Pregunta de Investigación

> ¿En qué medida la estabilización de la varianza mediante transformaciones logarítmicas mejora la capacidad predictiva de los modelos SARIMA y Prophet para el pronóstico de caudales diarios en la cuenca del río Las Ceibas, minimizando el error en periodos de transición climática?

**Respuesta a la Pregunta de Investigación:**

No se puede responder directamente a la pregunta principal en el contexto de este proyecto, ya que **no se aplicaron transformaciones logarítmicas** a la serie de caudal para estabilizar la varianza, ni se realizó un análisis específico en periodos de transición climática. Por lo tanto, no se evaluó en qué medida dicha estabilización mejoraría la capacidad predictiva de los modelos SARIMA y Prophet bajo esas condiciones. Los modelos SARIMA, Holt-Winters y Prophet se aplicaron directamente a la serie original (imputada y con *capping*) y su desempeño se evaluó en función de esa entrada.

**Preguntas Orientadoras (Objetivos):** Para dar respuesta a la pregunta central, planteamos los siguientes interrogantes técnicos:

*   **¿Estabilidad?**: ¿Logra la transformación logarítmica reducir significativamente el Coeficiente de Variación mensual en comparación con la serie original?
    *   **Respuesta (basada en el proyecto):** En este proyecto, no se aplicó una transformación logarítmica a la serie de caudal. Por lo tanto, no se evaluó si esta transformación reduciría significativamente el Coeficiente de Variación mensual. Los modelos se ajustaron directamente a la serie original, asumiendo una varianza aceptable para los propósitos del pronóstico.

*   **¿Estacionariedad?**: ¿Qué nivel de diferenciación (\( d \)) es necesario aplicar a la serie transformada para alcanzar el consenso de estacionariedad en las pruebas ADF y KPSS?
    *   **Respuesta (basada en el proyecto):** Aunque no se trabajó con una serie log-transformada, para el modelo SARIMA se utilizó una diferenciación de orden 1 (\( d=1 \)) para la parte no estacional y una diferenciación estacional de orden 1 (\( D=1 \)) con un período de 30 días (\( s=30 \)). Estos órdenes de diferenciación fueron seleccionados para inducir la estacionariedad de la serie original, lo cual es un requisito para el ajuste de los modelos ARIMA/SARIMA.

*   **¿Precisión?**: ¿Cuál de los modelos (SARIMA o Prophet) presenta un menor error cuadrático medio (RMSE) al ser evaluado sobre la serie con varianza estabilizada?
    *   **Respuesta (basada en el proyecto):** En este proyecto, el modelo **SARIMA** presentó el menor Error Cuadrático Medio (RMSE) en el conjunto de prueba (0.49 m³/s), superando a Holt-Winters (0.57 m³/s) y Prophet (1.47 m³/s). Es importante reiterar que esta evaluación se realizó sobre la serie original y no sobre una serie con varianza estabilizada mediante transformación logarítmica.

Variable objetivo: Caudal medio diario (m³/s)
Horizonte de pronóstico: 30 días
Métrica principal: RMSE (Root Mean Squared Error)

### Limitaciones identificadas:
-   **Falta de Transformación de Varianza**: La no aplicación explícita de transformaciones logarítmicas (o Box-Cox) para estabilizar la varianza podría limitar la capacidad de los modelos para capturar dinámicas más complejas, especialmente si la variabilidad de la serie de caudal cambia significativamente con el nivel del caudal. Esto podría ser un factor, aunque no crítico en este caso, en períodos de estiaje o crecidas extremas.
-   **Horizonte Estacional Aproximado para SARIMA**: El uso de un período estacional de 30 días para SARIMA, en lugar de 365 días (el ciclo anual real), es una simplificación computacional. Si bien esto hace el modelo viable, podría no capturar completamente la estacionalidad anual más sutil o de largo plazo que podría tener la serie.
-   **Ausencia de Variables Exógenas**: Los modelos se basan únicamente en la información histórica del propio caudal. La incorporación de variables exógenas como la precipitación, la temperatura o eventos climáticos (El Niño/La Niña) podría mejorar significativamente la capacidad predictiva y la robustez de los pronósticos.
-   **Análisis Limitado de Períodos de Transición Climática**: No se realizó un análisis específico ni se adaptaron los modelos para los períodos de transición climática, que podrían exhibir patrones de comportamiento del caudal diferentes o más complejos.
-   **Validación Cruzada Simplificada**: La metodología de evaluación se limitó a un enfoque de 'Hold-Out' simple (división train/test fija), lo que puede no ser tan robusto como una validación cruzada temporal ('walk-forward validation') para evaluar la estabilidad del modelo a lo largo del tiempo.

### Recomendación:

Se recomienda la implementación del modelo **SARIMA(1,1,1)(1,1,1,30)** para el pronóstico del caudal medio diario en la estación Pueblo Nuevo con un horizonte de 30 días, dado su desempeño superior en las métricas de error (RMSE, MAE, MAPE). Para mejorar la robustez y precisión futura, se sugiere explorar la incorporación de variables climáticas exógenas (como precipitación) y considerar una validación cruzada temporal más exhaustiva. Además, se podría evaluar el impacto de una transformación logarítmica de la serie en la estabilidad de la varianza, especialmente si los errores se distribuyen de manera heterocedástica. Finalmente, se recomienda realizar un ajuste fino de los hiperparámetros de SARIMA (p, d, q, P, D, Q) utilizando técnicas de búsqueda de cuadrícula o optimización bayesiana para maximizar aún más su rendimiento.

### Sugerencias para Mejorar el Proyecto:

Para abordar las limitaciones y mejorar la capacidad predictiva de tus modelos, podrías considerar los siguientes pasos:

## 11. Conclusiones del Proyecto (Semanas 1-4)

### Resumen del Pipeline Completo

| Semana | Notebook | Objetivo | Resultado Clave |
|--------|----------|----------|----------------|
| **1** | `01_carga_exploracion_caudal` | Carga, exploración y diagnóstico inicial | Serie de ~2,900 registros (2010-2017), estación Pueblo Nuevo, con gaps identificados |
| **2** | `02_tratamiento_nan_imputacion_caudal` | Limpieza, imputación y transformación | Serie diaria completa (`caudal_limpio_diario.csv`), interpolación lineal, capping P1-P99 |
| **3** | `03_eda_avanzado_estacionariedad_caudal` | EDA avanzado, descomposición y estacionariedad | Estacionalidad anual confirmada (STL), serie no estacionaria (ADF/KPSS), régimen bimodal |
| **4** | `04_forecasting_sarima_hw_prophet_caudal` | Pronóstico con SARIMA, Holt-Winters y Prophet | Modelo óptimo identificado, pronóstico a 30 días generado |

### Hallazgos Principales

1. **Calidad de datos**: La estación PUEBLO NUEVO presenta datos con gaps intermitentes que fueron exitosamente imputados mediante interpolación lineal, preservando la estructura temporal.

2. **Patrón hidrológico**: El caudal exhibe una marcada estacionalidad anual con régimen bimodal (dos picos), coherente con el patrón de precipitación de la zona andina colombiana.

3. **Capacidad predictiva**: Los tres modelos evaluados logran capturar la tendencia general del caudal, siendo el de menor RMSE el más adecuado para pronósticos operacionales.

4. **Valor práctico**: El pipeline desarrollado es replicable para cualquier estación del sistema DHIME/IDEAM, contribuyendo a la gestión de recursos hídricos en Colombia.

### Trabajo Futuro
- Incorporar variables climáticas exógenas (precipitación, temperatura).
- Explorar modelos de aprendizaje profundo (LSTM, GRU).
- Implementar validación cruzada temporal con ventanas deslizantes.
- Extender el análisis a múltiples estaciones para comparación regional.

---
**Proyecto desarrollado con datos del DHIME – IDEAM Colombia**  
*Estación 21117100 – PUEBLO NUEVO | Caudal Medio Diario (m³/s) | 2010–2017*

## 12. Incorporación de Variables Exógenas: Precipitación

Para mejorar la capacidad predictiva de nuestro modelo, especialmente SARIMA, podemos incorporar variables exógenas que influyan en el caudal, como la **precipitación**.

### 12.1. Cargar y Preparar Datos de Precipitación

In [80]:
# =============================================================================
# CONCEPTO: Carga y preparación de datos exógenos (precipitación)
# -----------------------------------------------------------------------------
# Cargar un archivo CSV con datos de precipitación diaria.
# Es crucial que la columna de fecha esté en el mismo formato y que la frecuencia
# sea la misma (diaria) que la serie de caudal.
# Se asume que el archivo 'precipitacion_diaria.csv' existe y tiene una columna
# 'Precipitacion' y 'Fecha'.
#
# Pasos clave:
# 1. Cargar el CSV, asegurando que 'Fecha' sea el índice y se parseen las fechas.
# 2. Asegurar la frecuencia diaria (`.asfreq("D")`).
# 3. Rellenar valores faltantes en la precipitación, ya que los modelos no los toleran.
#    `fillna(0)` es una estrategia común para precipitación, asumiendo 0 si no hay registro.
# 4. Alinear los datos de precipitación con el DataFrame principal de caudal (`df`).
#    Esto es crucial para que los índices de tiempo coincidan perfectamente.
#
# CRITERIO DE USO: Los datos exógenos deben tener la misma periodicidad y rango
# de fechas que la serie endógena (caudal). El manejo de NaNs en los exógenos
# es tan importante como en la serie principal.
# =============================================================================
import plotly.express as px # Importado aquí para robustez

# --- RECARGAR df ORIGINAL PARA EVITAR DUPLICACIÓN DE COLUMNAS ---
# Esto asegura que df no tenga ya la columna 'Precipitacion' de ejecuciones previas
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DATOS/SEMANA4/caudal_eda_avanzado.csv", index_col="Fecha", parse_dates=True)
df = df.asfreq("D")
# ------------------------------------------------------------------

# Cargar serie de precipitación (asumiendo que existe un archivo similar)
# df_prec = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/DATOS/SEMANA4/precipitacion_diaria.csv", index_col="Fecha", parse_dates=True)

# --- MOCKUP DE DATOS DE PRECIPITACIÓN PARA EJEMPLO ---
# Si no tienes un archivo de precipitación real, podemos crear un mockup
# para demostrar la funcionalidad.

print("🔄 Generando datos de precipitación de ejemplo...")
# Crear una serie de precipitación de ejemplo con la misma duración que 'df'
np.random.seed(42) # para reproducibilidad
df_prec = pd.DataFrame(
    {
        "Precipitacion": np.random.rand(len(df)) * 20 # Valores de precipitación en mm
    },
    index=df.index
)
# Añadir algo de estacionalidad básica al mockup
df_prec["Precipitacion"] = df_prec["Precipitacion"] + np.sin(np.arange(len(df)) * 2 * np.pi / 365) * 10
# Asegurar que no haya valores negativos por el sin()
df_prec["Precipitacion"] = df_prec["Precipitacion"].apply(lambda x: max(0, x))
# ------------------------------------------------------

# Asegurar frecuencia diaria
df_prec = df_prec.asfreq("D")

# Rellenar NaNs (si los hubiera en datos reales) – Para el mockup, no debería haber NaNs
df_prec["Precipitacion"] = df_prec["Precipitacion"].fillna(0)

# Alinear con el DataFrame principal 'df'
# Esto asegura que tengamos las mismas fechas y longitud
df = df.join(df_prec, how='left')

print(f"✅ Datos de precipitación cargados y alineados")
print(f"   Registros de precipitación: {len(df_prec)}")
print(f"   NaN en precipitación (después de fillna): {df['Precipitacion'].isna().sum()}")
display(df[['Caudal', 'Precipitacion']].head())
display(df[['Caudal', 'Precipitacion']].tail())

fig = px.line(df, x=df.index, y="Precipitacion", title="Precipitación Diaria de Ejemplo")
fig.update_layout(yaxis_title="Precipitación (mm)", width=900, height=400)
fig.show()

🔄 Generando datos de precipitación de ejemplo...
✅ Datos de precipitación cargados y alineados
   Registros de precipitación: 2922
   NaN en precipitación (después de fillna): 0


,Caudal,Precipitacion
Fecha,,
2010-01-01,1.196,7.490802
2010-01-02,1.196,19.186420
2010-01-03,1.196,14.984095
2010-01-04,1.196,12.489366
2010-01-05,1.196,3.808397


,Caudal,Precipitacion
Fecha,,
2017-12-27,2.655625,16.452311
2017-12-28,2.287125,8.108368
2017-12-29,2.480417,5.886485
2017-12-30,2.655625,6.505903
2017-12-31,2.299167,14.424560


### 12.2. División Train / Test para Datos Exógenos

Ahora que tenemos la precipitación en el DataFrame principal, debemos dividirla en conjuntos de entrenamiento y prueba, de la misma manera que hicimos con el caudal.

In [81]:
# =============================================================================
# CONCEPTO: División train/test para la variable exógena
# -----------------------------------------------------------------------------
# La variable exógena ('Precipitacion') debe dividirse usando el mismo punto de
# corte temporal que la serie endógena ('Caudal').
#
# train_exog = df["Precipitacion"].iloc[:-HORIZONTE]
# test_exog  = df["Precipitacion"].iloc[-HORIZONTE:]
#   → Usa el mismo 'HORIZONTE' definido previamente (60 días).
#
# Es fundamental que `train_exog` tenga la misma longitud y fechas que `train`,
# y `test_exog` la misma que `test`.
#
# CRITERIO DE USO: La coherencia entre los conjuntos de datos exógenos y endógenos
# (en cuanto a fechas y longitud) es vital para el correcto ajuste y pronóstico
# de los modelos que aceptan variables exógenas (como SARIMAX).
# =============================================================================

# Re-definir HORIZONTE, train y test para robustez, asumiendo que `df` está disponible.
HORIZONTE = 60  # días de test (debe coincidir con la definición original)
train = df["Caudal"].iloc[:-HORIZONTE]
test = df["Caudal"].iloc[-HORIZONTE:]

# División de la variable exógena (precipitación)
train_exog = df["Precipitacion"].iloc[:-HORIZONTE]
test_exog = df["Precipitacion"].iloc[-HORIZONTE:]

print(f"✅ Variable exógena dividida")
print(f"   Train exógeno: {len(train_exog)} días")
print(f"   Test exógeno:  {len(test_exog)} días")

# Verificar alineación (opcional pero buena práctica)
print(f"   Alineación train: {train.index.equals(train_exog.index)}")
print(f"   Alineación test:  {test.index.equals(test_exog.index)}")

✅ Variable exógena dividida
   Train exógeno: 2862 días
   Test exógeno:  60 días
   Alineación train: True
   Alineación test:  True


### 12.3. Modelo SARIMAX con Variable Exógena

Ahora podemos entrenar un modelo SARIMAX, utilizando la precipitación como variable exógena (`exog`).

**SARIMAX** (*Seasonal AutoRegressive Integrated Moving Average with eXogenous variables*) es una extensión natural del modelo SARIMA que nos permite incorporar otras series temporales que pueden explicar el comportamiento de nuestra serie principal.

In [82]:
%%time
from statsmodels.tsa.statespace.sarimax import SARIMAX # Importado aquí para robustez

# =============================================================================
# CONCEPTO: Entrenamiento del modelo SARIMAX con variable exógena
# -----------------------------------------------------------------------------
# SARIMAX(endog, exog, order, seasonal_order, ...)
#   - exog  → la variable exógena de entrenamiento (`train_exog`)
#             Debe tener la misma longitud y fechas que `endog` (`train`)
#
# .fit(disp=False, maxiter=200)
#   → El proceso de ajuste es el mismo que para SARIMA, pero ahora el modelo
#     considerará la influencia de la variable 'Precipitacion' al estimar sus parámetros.
#
# La interpretación de los coeficientes de `exog` en el resumen del modelo
# indicará la relación entre la precipitación y el caudal, manteniendo el resto
# de los componentes (AR, MA, estacional) del modelo.
#
# CRITERIO DE USO: La inclusión de variables exógenas debe estar respaldada por
# una hipótesis teórica (ej. precipitación influye en caudal) y verificarse
# que mejora el ajuste y el pronóstico del modelo (menor AIC/BIC, mejores métricas en test).
# =============================================================================

# Re-definir HORIZONTE, train y train_exog para robustez, asumiendo que `df` está disponible.
HORIZONTE = 60 # días de test (debe coincidir con la definición original)
train = df["Caudal"].iloc[:-HORIZONTE]
train_exog = df["Precipitacion"].iloc[:-HORIZONTE]

print("🔄 Entrenando SARIMAX(1,1,1)(1,1,1,30) con Precipitación como exógena...")

modelo_sarimax = SARIMAX(
    train,
    exog=train_exog, # Aquí se pasa la variable exógena
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 30),
    enforce_stationarity=False,
    enforce_invertibility=False,
)
resultado_sarimax = modelo_sarimax.fit(disp=False, maxiter=200)

print(f"\n✅ SARIMAX con exógena entrenado")
print(f"   AIC: {resultado_sarimax.aic:.2f}")
print(f"   BIC: {resultado_sarimax.bic:.2f}")
print(f"\n📋 Resumen de parámetros (incluyendo coef. exógena):")
print(resultado_sarimax.summary().tables[1])

🔄 Entrenando SARIMAX(1,1,1)(1,1,1,30) con Precipitación como exógena...

✅ SARIMAX con exógena entrenado
   AIC: 5327.01
   BIC: 5362.63

📋 Resumen de parámetros (incluyendo coef. exógena):
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Precipitacion     0.0014      0.002      0.780      0.435      -0.002       0.005
ar.L1             0.5812      0.014     42.872      0.000       0.555       0.608
ma.L1            -0.8701      0.010    -83.434      0.000      -0.891      -0.850
ar.S.L30         -0.0364      0.021     -1.705      0.088      -0.078       0.005
ma.S.L30         -1.0199      0.010   -105.830      0.000      -1.039      -1.001
sigma2            0.3633      0.006     60.960      0.000       0.352       0.375
CPU times: user 1min 15s, sys: 649 ms, total: 1min 16s
Wall time: 1min 10s


### 12.4. Pronóstico SARIMAX con Variable Exógena

Para pronosticar con un modelo SARIMAX que incluye variables exógenas, es crucial proporcionar los valores futuros de la variable exógena. En este caso, necesitaríamos un pronóstico de precipitación para los 60 días del período de prueba (`test_exog`).

In [83]:
# =============================================================================
# CONCEPTO: Pronóstico SARIMAX sobre el período test con variable exógena
# -----------------------------------------------------------------------------
# resultado_sarimax.forecast(steps=HORIZONTE, exog=test_exog)
#   - exog  → los valores de la variable exógena para el horizonte de pronóstico.
#             En este caso, `test_exog` contiene la precipitación de los 60 días
#             del período de prueba. Si fuera un pronóstico a futuro real, necesitaríamos
#             un *pronóstico* de precipitación para esos 30 días futuros.
#
# Es vital que la longitud de `exog` en `forecast()` coincida con `steps`.
#
# CRITERIO DE USO: La disponibilidad de pronósticos fiables para las variables
# exógenas es un factor limitante para SARIMAX. Si no tenemos un buen pronóstico
# de precipitación, la ventaja de usar SARIMAX con exógenas disminuye o puede
# incluso empeorar el pronóstico.
# =============================================================================
import numpy as np # Importado aquí para robustez
import plotly.graph_objects as go # Importado aquí para robustez
from plotly.subplots import make_subplots # Importado aquí para robustez
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error # Importado aquí para robustez

# Re-definir HORIZONTE, test y test_exog para robustez, asumiendo que `df` está disponible.
HORIZONTE = 60 # días de test (debe coincidir con la definición original)
test = df["Caudal"].iloc[-HORIZONTE:]
test_exog = df["Precipitacion"].iloc[-HORIZONTE:]

# Pronóstico SARIMAX sobre el período de test, usando la precipitación del test como exógena
# Esta línea asume que `resultado_sarimax` ha sido definido en la celda anterior y se ejecutó con éxito.
pred_sarimax_exog = resultado_sarimax.forecast(steps=HORIZONTE, exog=test_exog)
pred_sarimax_exog.index = test.index

# Diagnóstico de residuos
residuos_sarimax_exog = resultado_sarimax.resid

fig_sarimax_diag = make_subplots(rows=1, cols=2,
                    subplot_titles=["Pronóstico SARIMAX con Exógena vs Real", "Residuos del modelo con Exógena"])

fig_sarimax_diag.add_trace(go.Scatter(x=test.index, y=test, mode="lines",
              name="Real", line=dict(color="#1f77b4", width=2)), row=1, col=1)
fig_sarimax_diag.add_trace(go.Scatter(x=test.index, y=pred_sarimax_exog, mode="lines",
              name="SARIMAX + Precipitación", line=dict(color="#8c564b", width=2, dash="dash")), row=1, col=1)

fig_sarimax_diag.add_trace(go.Histogram(x=residuos_sarimax_exog, nbinsx=50,
              marker_color="#8c564b", name="Residuos"), row=1, col=2)

fig_sarimax_diag.update_layout(title="Modelo SARIMAX(1,1,1)(1,1,1,30) con Precipitación",
                  width=1000, height=400, showlegend=True)
fig_sarimax_diag.update_yaxes(title_text="Caudal (m³/s)", row=1, col=1)
fig_sarimax_diag.show()

rmse_sx = np.sqrt(mean_squared_error(test, pred_sarimax_exog))
mae_sx = mean_absolute_error(test, pred_sarimax_exog)
mape_sx = mean_absolute_percentage_error(test, pred_sarimax_exog) * 100
print(f"\n📊 SARIMAX + Precipitación – Métricas en Test:")
print(f"   RMSE: {rmse_sx:.2f} m³/s")
print(f"   MAE:  {mae_sx:.2f} m³/s")
print(f"   MAPE: {mape_sx:.1f}%")



📊 SARIMAX + Precipitación – Métricas en Test:
   RMSE: 0.50 m³/s
   MAE:  0.45 m³/s
   MAPE: 18.2%
